Structured Output: # Pydantic 

In [35]:
import os
from langchain.chat_models import init_chat_model

os.environ['LLAMA_API_KEY'] = os.getenv("LLAMA_API_KEY")
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.7', 'langchain': '1.4.10'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x319880350>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x3198db650>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie on a scale of 1 to 10")

In [10]:
model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.7', 'langchain': '1.4.10'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x1682e5590>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1682e5f90>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movi

In [8]:
model.invoke("Please provide the details of the movie 'Inception'")

AIMessage(content='**Inception (2010)**  \n\n| Category | Details |\n|----------|---------|\n| **Director** | Christopher Nolan |\n| **Screenwriter** | Christopher Nolan |\n| **Producer(s)** | Christopher Nolan, Emma Thomas, Emma Thomas (co‑producer) |\n| **Production Company** | Syncopy, Legendary Pictures |\n| **Distributor** | Warner Bros. Pictures |\n| **Release Dates** | • United States – 16 July 2010 (limited) <br>• United States – 15 July 2010 (wide) <br>• International – 18–20 July 2010 (various dates) |\n| **Runtime** | 2\u202fh\u202f28\u202fmin |\n| **Country** | United States |\n| **Language** | English |\n| **Budget** | ≈\u202f$160\u202fmillion |\n| **Box‑office** | $829\u202fmillion worldwide |\n| **Genres** | Science‑fiction, Action, Thriller, Adventure |\n| **MPAA Rating** | PG‑13 (U.S.) |\n\n---\n\n### Plot (≈\u202f250 words)\n\nDom Cobb (Leonardo DiCaprio) is a thief who specializes in “extraction”—stealing secrets from deep within the subconscious during the dream sta

In [11]:
model_with_structured_output.invoke("Please provide details about the movie 'Inception'")


Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

Message output with parsed structure

In [13]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(...,description="The title of the movie")
    year: int = Field(...,description="The year the movie was released")
    director: str = Field(...,description="The director of the movie")
    rating: float = Field(...,description="The rating of the movie on a scale of 1 to 10")

model_with_structured_output = model.with_structured_output(Movie, include_raw=True)
model_with_structured_output.invoke("Please provide details about the movie 'Inception'")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use function. Provide details. So call functions.Movie with director Christopher Nolan, rating maybe 8.8, title Inception, year 2010.', 'tool_calls': [{'id': 'fc_c60ec394-f581-4aeb-af84-d42057edf8b4', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 168, 'total_tokens': 242, 'completion_time': 0.085237222, 'completion_tokens_details': {'reasoning_tokens': 35}, 'prompt_time': 0.008103274, 'prompt_tokens_details': None, 'queue_time': 0.861348255, 'total_time': 0.093340496}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_996f667773', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1d461-2da8-73e9-ac78-038300e0108a-0', tool_calls=[{'name': 'Movie', 'a

Nested Structure

In [15]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    actors: list[Actor]
    genre: list[str]
    budget: float | None = Field(default=None, description="The budget of the movie in USD")

model_with_structured_output = model.with_structured_output(MovieDetails)
response = model_with_structured_output.invoke("Please provide details about the movie 'Inception'")
response

MovieDetails(title='Inception', year=2010, actors=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genre=['Action', 'Adventure', 'Sci-Fi'], budget=160000000.0)

TypeDict


typdict provides a simpler alternative using python built in typing, ideal who don't need untime validation

In [19]:
from typing_extensions import Annotated, TypedDict

class MovieDict(TypedDict):
    title: Annotated[str, ...,"The title of the movie"]
    year: Annotated[int, ...,"The year the movie was released"]
    director: Annotated[str, ...,"The director of the movie"]
    rating: Annotated[float, ...,"The rating of the movie on a scale of 1 to 10"]


model_with_typedict_output = model.with_structured_output(MovieDict)
response = model_with_typedict_output.invoke("Please provide details about the movie 'Inception'")
response

{'director': 'Christopher Nolan',
 'rating': 8.8,
 'title': 'Inception',
 'year': 2010}

In [17]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    actors: list[Actor]
    genre: list[str]
    budget: float | None = Field(default=None, description="The budget of the movie in USD")

model_with_structured_output = model.with_structured_output(MovieDetails)
response = model_with_structured_output.invoke("Please provide details about the movie 'Inception'")
response

{'actors': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'}],
 'budget': 160000000,
 'genre': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Inception',
 'year': 2010}

Data Classes

- data class is a class basically containing data although they are not really any restriction

In [ ]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field


class ContactInfo(BaseModel):
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email address of the contact")
    phone: str = Field(..., description="The phone number of the contact")

agent = create_agent(
    model= 'groq:openai/gpt-oss-20b',
    response_format= ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract info from: John Doe, john.doe@example.com, 123-456-7890"}]
})
result
print(result["structured_response"])
# {'name': 'John Doe', 'email': 'john.doe@example.com', 'phone': '123-456-7890'}


name='John Doe' email='john.doe@example.com' phone='123-456-7890'


In [ ]:
# with typedict
from langchain.agents import create_agent
from typing_extensions import Annotated, TypedDict

class ContactInfo(TypedDict):
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email address of the contact")
    phone: str = Field(..., description="The phone number of the contact")

agent = create_agent(
    model= 'groq:openai/gpt-oss-20b',
    response_format= ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract info from: John Doe, john.doe@example.com, 123-456-7890"}]
})
result
print(result["structured_response"])
# {'name': 'John Doe', 'email': 'john.doe@example.com', 'phone': '123-456-7890'}


{'name': 'John Doe', 'email': 'john.doe@example.com', 'phone': '123-456-7890'}


In [ ]:
# Dataclass 

from langchain.agents import create_agent
from dataclasses import dataclass 

@dataclass
class ContactInfo:
    """ A dataclass to hold contact information extracted from text."""
    name: str
    email: str
    phone: str

agent = create_agent(
    model= 'groq:openai/gpt-oss-20b',
    response_format= ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract info from: John Doe, john.doe@example.com, 123-456-7890"}]
})
result
print(result["structured_response"])
# ContactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')

ContactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')
